Note: Due to API quota limitations on the OpenAI account, the Hugging Face embedding model "sentence-transformers/all-MiniLM-L6-v2" was used to demonstrate the same embedding workflow.

Mentioned this change in readme file also. I want to transparent about this thats whyy.

In [27]:
#importing the libraries

#i met with the error that I didnt have installed the package, so im going it again

!pip install -q langchain-huggingface
!pip install -q langchain-community
!pip install -q pypdf
!pip install -q faiss-cpu
!pip install -q chromadb

from langchain_huggingface import HuggingFaceEmbeddings
import time

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [14]:
#loading the embedding model from huggingface

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [15]:
#faced the error that I forgot to add the files, so im adding again and checking using this commands

import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'employees.csv', 'DevAnand_Resume.pdf', 'notes.txt', 'sample_data']


In [16]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Re-loading documents (originally from deleted cell d8OxBNQCoJLp)
documents = []
documents.extend(TextLoader("notes.txt").load())
documents.extend(PyPDFLoader("DevAnand_Resume.pdf").load())
documents.extend(CSVLoader("employees.csv").load())

# Re-splitting documents (originally from deleted cell dBoN-w72r9o8)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
document_chunks = text_splitter.split_documents(documents)

# Generating embedding for the first document chunk
#just a try for the demo to check whether it works or not and then we can generate embedd for all the chunks
embedding = embedding_model.embed_query(document_chunks[0].page_content)

print("Embedding Vector Length:", len(embedding))

print("\nFirst 10 Values:")
print(embedding[:10])

Embedding Vector Length: 384

First 10 Values:
[-0.028264539316296577, -0.06470387428998947, 0.041302576661109924, 0.023598013445734978, 0.02760906331241131, 0.016336580738425255, -0.04025304317474365, -0.0010839167516678572, 0.026831351220607758, -0.0011767823016270995]


In [17]:
#'above cell worked, so im gonna perform embedding for all the chunks
start_time = time.time()

all_embeddings = embedding_model.embed_documents([chunk.page_content for chunk in document_chunks])
end_time = time.time()

print("Total Embeddings Generated:", len(all_embeddings))
print("Embedding Dimension:", len(all_embeddings[0]))
print(f"Time Taken: {end_time - start_time:.2f} seconds")

Total Embeddings Generated: 17
Embedding Dimension: 384
Time Taken: 0.32 seconds


In [18]:
#verifing the embeddings in this cell
print("\nSample Embedding Values:")
print(all_embeddings[0][:10])

print("\nFirst Chunk Preview:")
print(document_chunks[0].page_content[:200])


Sample Embedding Values:
[-0.028264528140425682, -0.06470388174057007, 0.04130260646343231, 0.02359798550605774, 0.027609026059508324, 0.016336588189005852, -0.04025302454829216, -0.0010839245514944196, 0.026831315830349922, -0.0011768067488446832]

First Chunk Preview:
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses n


### Observation

- Successfully generated embeddings for all document chunks using the Hugging Face model.
- Each chunk is represented as a 384-dimensional vector.
- These embeddings will be stored in a vector database for similarity search in the next tasks.

In [19]:
# Task 2 - Hugging Face Embedding Model

print("Model Name : sentence-transformers/all-MiniLM-L6-v2")
print("Embedding Dimension :", len(all_embeddings[0]))
print("Total Embeddings :", len(all_embeddings))

print("Sample Embedding Values:")
print(all_embeddings[0][:10])

Model Name : sentence-transformers/all-MiniLM-L6-v2
Embedding Dimension : 384
Total Embeddings : 17
Sample Embedding Values:
[-0.028264528140425682, -0.06470388174057007, 0.04130260646343231, 0.02359798550605774, 0.027609026059508324, 0.016336588189005852, -0.04025302454829216, -0.0010839245514944196, 0.026831315830349922, -0.0011768067488446832]


#Task 3 – OpenAI vs Hugging Face Comparison
1. When should we prefer OpenAI Embeddings?

- Better semantic understanding.
- High-quality embeddings for production applications.
- Suitable for Retrieval-Augmented Generation (RAG) systems.
- Requires an internet connection and API credits.

2. When should we prefer Hugging Face Embeddings?

- Works offline after downloading the model.
- Free to use.
- Suitable for learning, local development and small projects.
- No API cost.

3. Cost vs Performance

| OpenAI | Hugging Face
|---------|--------------
| Paid API | Free
| Better semantic quality | Good semantic quality
| Cloud-based | Runs locally
| Internet required | Offline after download
| Best for production | Best for development and learning



In [22]:
# Task 4 - Document Similarity Search

from langchain_community.vectorstores import FAISS

# Creating FAISS vector store using this line of codeing
vectorstore = FAISS.from_documents(document_chunks, embedding_model)
print("FAISS Vector Store Created Successfully")

#using the searvh functionality
def search(query, top_k=3):
    results = vectorstore.similarity_search(query, k=top_k)
    print(f"\nQuery: {query}\n")
    for i, doc in enumerate(results, start=1):
        print(f"Result {i}:")
        print(doc.page_content[:250])
        print("-" * 50)

#giving the qreries, if it is present in files means, it will come oterwise it will not come

search("Artificial Intelligence")
search("Python")
search("Machine Learning")
search("Skills")
search("Experience")
search("Employee")


FAISS Vector Store Created Successfully

Query: Artificial Intelligence

Result 1:
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Result 2:
• AI/ML: YOLOv8, OpenCV, NLP, Computer Vision, Model Training & Inference  
ACHIEVEMENTS 
• Invited as Resource Person by Coimbatore Institute of Technology (CIT) to conduct a hands -on 'Vibe Coding' workshop for 
150+ students (Sep 13, 2025); receiv
--------------------------------------------------
Result 3:
auto-scaling, ensuring high availability and sub-second response times.  
• Tech: AWS Comprehend Medical, Bedrock, Personalize, Lambda, API Gatewa y, DynamoDB, S3, Amplify, CloudWatch. 
TECHNICAL SKILLS 
• Programming: Python, Go, Java    
• Database
------------------------------------------

Observation is that the similarity search returned the most relevant document chunks for each query, FAISS compares embeddings instead of exact keywords, enabling semantic search

In [23]:
# Task 5 - Similarity Search using LangChain
query = "Artificial Intelligence"
results = vectorstore.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Document {i}")
    print(doc.page_content[:250])
    print("-" * 50)

Query: Artificial Intelligence

Document 1
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Document 2
• AI/ML: YOLOv8, OpenCV, NLP, Computer Vision, Model Training & Inference  
ACHIEVEMENTS 
• Invited as Resource Person by Coimbatore Institute of Technology (CIT) to conduct a hands -on 'Vibe Coding' workshop for 
150+ students (Sep 13, 2025); receiv
--------------------------------------------------
Document 3
auto-scaling, ensuring high availability and sub-second response times.  
• Tech: AWS Comprehend Medical, Bedrock, Personalize, Lambda, API Gatewa y, DynamoDB, S3, Amplify, CloudWatch. 
TECHNICAL SKILLS 
• Programming: Python, Go, Java    
• Database
--------------------------------------------------
Document 4
and take proactive

In [24]:
#using different different words or the qrueries
query = "Python"
results = vectorstore.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Document {i}")
    print(doc.page_content[:250])
    print("-" * 50)

query = "Machine Learning"
results = vectorstore.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Document {i}")
    print(doc.page_content[:250])
    print("-" * 50)

Query: Python

Document 1
an empathetic chatbot that delivers coping strategies and escalates high-risk cases to professional counselors. 
• Developed an anonymized admin analytics dashboard for institutional trend monitoring, plus confidential counselor booking, 
peer suppor
--------------------------------------------------
Document 2
auto-scaling, ensuring high availability and sub-second response times.  
• Tech: AWS Comprehend Medical, Bedrock, Personalize, Lambda, API Gatewa y, DynamoDB, S3, Amplify, CloudWatch. 
TECHNICAL SKILLS 
• Programming: Python, Go, Java    
• Database
--------------------------------------------------
Document 3
vehicle-based search; containerized the application using Docker for consistent, production -ready deployments. 
• Built a citizen portal for challan search, fine calculation, and payment integration; Tech: Python, FastAPI, YOLOv8, OpenCV, 
EasyOCR, 
--------------------------------------------------
Document 4
Artificial Intelligence (AI) enable

observation is that the LangChain provides a simple abstraction for similarity search and Only the query is required to retrieve the most relevant document chunks.

about the task 6:
Ollama embeddings require a locally running Ollama server and cannot be executed in Google Colab


In [25]:
# Task 7 - FAISS Vector Store

vectorstore.save_local("faiss_index")
print("FAISS index saved successfully.")

#loading the faisss indexes
from langchain_community.vectorstores import FAISS
loaded_vectorstore = FAISS.load_local("faiss_index",embedding_model,allow_dangerous_deserialization=True)
print("FAISS index loaded successfully.")

query = "Artificial Intelligence"

results = loaded_vectorstore.similarity_search(query)
#just verification
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content[:250])
    print("-" * 50)

query = "Python"
results = loaded_vectorstore.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content[:250])
    print("-" * 50)

FAISS index saved successfully.
FAISS index loaded successfully.
Query: Artificial Intelligence

Result 1
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Result 2
• AI/ML: YOLOv8, OpenCV, NLP, Computer Vision, Model Training & Inference  
ACHIEVEMENTS 
• Invited as Resource Person by Coimbatore Institute of Technology (CIT) to conduct a hands -on 'Vibe Coding' workshop for 
150+ students (Sep 13, 2025); receiv
--------------------------------------------------
Result 3
auto-scaling, ensuring high availability and sub-second response times.  
• Tech: AWS Comprehend Medical, Bedrock, Personalize, Lambda, API Gatewa y, DynamoDB, S3, Amplify, CloudWatch. 
TECHNICAL SKILLS 
• Programming: Python, Go, Java    
• Database
---------------------

Observation is that the FAISS index was successfully saved to the local directory and after reloading the index, similarity search returned the expected document chunks.

In [28]:
# Task 8 - ChromaDB Vector Store
#creating the chromadb using this codelines
from langchain_community.vectorstores import Chroma

# Creating ChromaDB vector store
chroma_db = Chroma.from_documents(document_chunks,embedding_model,persist_directory="chroma_db")
print("ChromaDB created successfully.")


# Saving ChromaDB to disk
chroma_db.persist()
print("ChromaDB saved successfully.")

ChromaDB created successfully.
ChromaDB saved successfully.


/tmp/ipykernel_513/456292401.py:11: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma_db.persist()


In [29]:
query = "Artificial Intelligence"
results = chroma_db.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content[:250])
    print("-" * 50)

Query: Artificial Intelligence

Result 1
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Result 2
• AI/ML: YOLOv8, OpenCV, NLP, Computer Vision, Model Training & Inference  
ACHIEVEMENTS 
• Invited as Resource Person by Coimbatore Institute of Technology (CIT) to conduct a hands -on 'Vibe Coding' workshop for 
150+ students (Sep 13, 2025); receiv
--------------------------------------------------
Result 3
auto-scaling, ensuring high availability and sub-second response times.  
• Tech: AWS Comprehend Medical, Bedrock, Personalize, Lambda, API Gatewa y, DynamoDB, S3, Amplify, CloudWatch. 
TECHNICAL SKILLS 
• Programming: Python, Go, Java    
• Database
--------------------------------------------------
Result 4
and take proactive steps t

In [30]:
#reloading the chroma db
loaded_chroma = Chroma(persist_directory="chroma_db",embedding_function=embedding_model)
print("ChromaDB loaded successfully.")

query = "Python"
results = loaded_chroma.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content[:250])
    print("-" * 50)

ChromaDB loaded successfully.
Query: Python

Result 1
an empathetic chatbot that delivers coping strategies and escalates high-risk cases to professional counselors. 
• Developed an anonymized admin analytics dashboard for institutional trend monitoring, plus confidential counselor booking, 
peer suppor
--------------------------------------------------
Result 2
auto-scaling, ensuring high availability and sub-second response times.  
• Tech: AWS Comprehend Medical, Bedrock, Personalize, Lambda, API Gatewa y, DynamoDB, S3, Amplify, CloudWatch. 
TECHNICAL SKILLS 
• Programming: Python, Go, Java    
• Database
--------------------------------------------------
Result 3
vehicle-based search; containerized the application using Docker for consistent, production -ready deployments. 
• Built a citizen portal for challan search, fine calculation, and payment integration; Tech: Python, FastAPI, YOLOv8, OpenCV, 
EasyOCR, 
--------------------------------------------------
Result 4
Artificial In

/tmp/ipykernel_513/1374231080.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  loaded_chroma = Chroma(persist_directory="chroma_db",embedding_function=embedding_model)


# Task 9 - FAISS vs ChromaDB Comparison
### 1. In-Memory vs Persistent Storage

**FAISS**
- Primarily stores vectors in memory.
- Index can be saved and loaded manually.

**ChromaDB**
- Stores vectors persistently on disk.
- Automatically manages the stored embeddings.

### 2. Use Cases for FAISS

- Fast similarity search
- Research and experimentation
- Local applications
- Large-scale vector indexing

### 3. Use Cases for ChromaDB

- Persistent document storage
- Retrieval-Augmented Generation (RAG)
- Semantic document search
- Production AI applications

In [31]:
# Task 10 - End-to-End Similarity Search Pipeline
def build_pipeline(embedding_backend="huggingface", vector_backend="faiss"):
    # Select embedding model
    if embedding_backend == "huggingface":
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    # Select vector database
    if vector_backend == "faiss":
        vector_db = FAISS.from_documents(document_chunks, embeddings)

    elif vector_backend == "chroma":
      vector_db = Chroma.from_documents(document_chunks,embeddings,persist_directory="pipeline_chroma")

    return vector_db

# Building the pipeline
pipeline = build_pipeline(embedding_backend="huggingface",vector_backend="faiss")
print("Pipeline created successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Pipeline created successfully.


In [32]:
query = "Artificial Intelligence"
results = pipeline.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content[:250])
    print("-" * 50)

#now testing the chroma db again
pipeline = build_pipeline(embedding_backend="huggingface",vector_backend="chroma")
results = pipeline.similarity_search("Python")
print("Results using ChromaDB\n")
for doc in results:
    print(doc.page_content[:200])
    print("-" * 50)

Query: Artificial Intelligence

Result 1
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Result 2
• AI/ML: YOLOv8, OpenCV, NLP, Computer Vision, Model Training & Inference  
ACHIEVEMENTS 
• Invited as Resource Person by Coimbatore Institute of Technology (CIT) to conduct a hands -on 'Vibe Coding' workshop for 
150+ students (Sep 13, 2025); receiv
--------------------------------------------------
Result 3
auto-scaling, ensuring high availability and sub-second response times.  
• Tech: AWS Comprehend Medical, Bedrock, Personalize, Lambda, API Gatewa y, DynamoDB, S3, Amplify, CloudWatch. 
TECHNICAL SKILLS 
• Programming: Python, Go, Java    
• Database
--------------------------------------------------
Result 4
and take proactive steps t

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Results using ChromaDB

an empathetic chatbot that delivers coping strategies and escalates high-risk cases to professional counselors. 
• Developed an anonymized admin analytics dashboard for institutional trend monitoring,
--------------------------------------------------
auto-scaling, ensuring high availability and sub-second response times.  
• Tech: AWS Comprehend Medical, Bedrock, Personalize, Lambda, API Gatewa y, DynamoDB, S3, Amplify, CloudWatch. 
TECHNICAL SKIL
--------------------------------------------------
vehicle-based search; containerized the application using Docker for consistent, production -ready deployments. 
• Built a citizen portal for challan search, fine calculation, and payment integration;
--------------------------------------------------
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses n
-------------------

Observation is that the pipeline successfully converted document chunks into embeddings and Embeddings were stored in both FAISS and ChromaDB

# Task 11 - Observations & Insights
### 1. Importance of Embeddings in Generative AI

Embeddings convert text into numerical vectors while preserving semantic meaning. They help AI systems understand the relationship between words, sentences, and documents, enabling semantic search and document retrieval.

### 2. Why Vector Databases are Required

Vector databases store embeddings efficiently and perform fast similarity searches. They retrieve the most relevant documents based on semantic similarity instead of exact keyword matching.

### 3. How This Pipeline Enables RAG Systems

The pipeline loads documents, splits them into chunks, converts each chunk into embeddings, stores them in a vector database, and retrieves the most relevant chunks for a user query. These retrieved chunks can then be provided to a Large Language Model (LLM) to generate accurate and context-aware responses.



Right now, the build_pipeline() function only supports Hugging Face